In [2]:
import chess
import random as rd

class Bot:
    name: str = "bot"

    def select_move(self, board):
        raise NotImplementedError

    def __repr__(self):
         return f"<{type(self).__name__} name={self.name!r}>"

class RandomBot(Bot):

    name = "random"

    def __init__(self, seed):
        self.rng = rd.Random(seed)

    def select_move(self, board):
        moves = list(board.legal_moves)
        if not moves:
            raise ValueError("no legal move available in this position")
        return self.rng.choice(moves)
    


In [9]:
import chess
b = chess.Board()
print(b)
print(len(list(b.legal_moves)))
b.push_san("e4")
print(b)
print(b.turn, b.fen())
bot = RandomBot(42)
a = bot.select_move(b)
print(a)
b.push(a)
print(b)

r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R
20
r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . P . . .
. . . . . . . .
P P P P . P P P
R N B Q K B N R
False rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1
b8a6
r . b q k b n r
p p p p p p p p
n . . . . . . .
. . . . . . . .
. . . . P . . .
. . . . . . . .
P P P P . P P P
R N B Q K B N R


In [15]:
pvalues: dict[chess.PieceType, int] = {
    chess.PAWN: 1,
    chess.KNIGHT: 3,
    chess.BISHOP: 3,
    chess.ROOK: 5,
    chess.QUEEN: 9,
    chess.KING: 0,
}



In [16]:
def ismate(board, move):
    board.push(move)
    r = board.is_checkmate()
    board.pop()
    return r

def move_gain(board, move):
    gain = 0
    if ismate(board,move):
        gain += 100
    if board.is_en_passant(move):
        gain += pvalues[chess.PAWN]

    else:
        taken = board.piece_at(move.to_square)
        if taken is not None:
            gain += pvalues[taken.piece_type]
        if move.promotion is not None:
            gain+= pvalues[move.promotion] - pvalues[chess.PAWN]
    return gain

In [19]:
class materialBot(Bot):

    name = "material"

    def __init__(self, seed):
        self.rng= rd.Random(seed)

    def select_move(self, board):
        moves = list(board.legal_moves) 
        if not moves:
            raise ValueError("no legal move available in this position")
        best = max(move_gain(board,m) for m in moves)
        best_moves = [m for m in moves if move_gain(board, m)==best]
        return self.rng.choice(best_moves)
    



In [ ]:
b = chess.Board()
print(b)
print(len(list(b.legal_moves)))
b.push_san("e4")

mat = materialBot(42)
b.push(mat.select_move(b))

rand = RandomBot(42)
for i in range(29):
    b.push(rand.select_move(b))
    b.push(mat.select_move(b))

print(b)
# mat wins against random





r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R
20
. . b . k b . r
p . . p p . . p
p q p . . . . p
. . . . . . . .
. . . . . . . .
. . . . . p . .
. . . . . . . .
. . . . . K . r
